# 02 資料處理與視覺化 — 參考解答

松柏護理之家退伍軍人症 line list 練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# Plotly: 確保在靜態建置（jupyter-book build）時也能輸出互動圖
pio.renderers.default = "notebook"


## 題目 1：讀入並檢視資料

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"資料維度：{df.shape[0]} 筆 × {df.shape[1]} 欄")
print(f"\n欄位名稱：{df.columns.tolist()}")
df.head()

In [ ]:
df.info()

## 題目 2：日期轉換與衍生變項

In [ ]:
# 日期轉換
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")

# 建立 infected 欄位
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 計算 onset_to_hosp_days
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# 印出前 10 位感染者
infected_df = df[df["infected"] == 1]
infected_df[["case_id", "symptom_onset_date", "onset_to_hosp_days"]].head(10)

## 題目 3：流行曲線

In [ ]:
import matplotlib.dates as mdates

cases = df[df["infected"] == 1]
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# 加入爆發前背景期（含零病例日）
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 題目 4：翼區侵襲率比較圖

In [ ]:
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("各翼區侵襲率比較")
ax.set_xlabel("翼區")
ax.set_ylabel("侵襲率 (%)")

for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 題目 5（挑戰題）：互動式分層流行曲線

In [ ]:
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    title="松柏護理之家退伍軍人症流行曲線（依樓層分層），2026 年 1 月",
    labels={"symptom_onset_date": "發病日期", "cases": "病例數", "floor": "樓層"},
)
fig.update_layout(
    bargap=0,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, rangemode="tozero"),
    plot_bgcolor="white",
)
fig.show()

### 解讀

- **流行曲線**：病例高峰集中在數天之內，呈現典型的 **共同暴露源（point source）** 型態
- **翼區比較**：3B 翼侵襲率最高，1B 翼最低 → 暴露源可能與特定區域設施有關
- **分層曲線**：若三樓流行高峰早於一樓，可能暗示暴露源在高樓層（例如水塔供水管路）

## 題目 6：頻率表與樞紐分析

In [ ]:
# 嚴重度次數分布
print("=== 嚴重度次數分布 ===")
print(df["clinical_severity"].value_counts())
print("\n=== 嚴重度百分比 ===")
print(df["clinical_severity"].value_counts(normalize=True).mul(100).round(1))

# 翼區 × 樓層 侵襲率表
print("\n=== 翼區 × 樓層 侵襲率 ===")
pivot = pd.pivot_table(
    df,
    values="infected",
    index="wing",
    columns="floor",
    aggfunc="mean",
    margins=True,
)
print(pivot.round(3))

## 題目 7：Method Chaining

In [ ]:
# 用 method chaining 一行完成：篩選 70+ 感染者 → 按樓層分組 → 算人數和死亡數 → 致死率 → 排序
result = (
    df
    .query("infected == 1 and age >= 70")
    .groupby("floor")
    .agg(
        n_cases=("case_id", "count"),
        n_deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .assign(cfr=lambda d: (d["n_deaths"] / d["n_cases"] * 100).round(1))
    .sort_values("cfr", ascending=False)
)
print("70 歲以上感染者，各樓層致死率：")
result

## 題目 8：合併資料與文字清理

In [ ]:
import numpy as np

# 1. 模擬實驗室檢驗資料
rng = np.random.default_rng(42)
infected_ids = df.loc[df["infected"] == 1, "case_id"].tolist()
lab = pd.DataFrame({
    "case_id": infected_ids,
    "ct_value": rng.uniform(15, 35, size=len(infected_ids)).round(1),
})
print(f"Lab 資料：{len(lab)} 筆")
lab.head()

In [ ]:
# 2. 合併 lab 資料
df_merged = pd.merge(df, lab, on="case_id", how="left")
print(f"合併後：{df_merged.shape}")
print(f"有 ct_value 的筆數：{df_merged['ct_value'].notna().sum()}")

# 3. 統一 wing 大小寫
df_merged["wing"] = df_merged["wing"].str.strip().str.upper()
print(f"\nWing 類別：{df_merged['wing'].unique()}")

# 4. 去除重複通報
before = len(df_merged)
df_merged = df_merged.drop_duplicates(subset="case_id", keep="first")
print(f"去重前：{before}，去重後：{len(df_merged)}")

# 5. 年齡最大的 5 位感染者
top5 = df_merged.query("infected == 1").nlargest(5, "age")
print("\n年齡最大的 5 位感染者：")
top5[["case_id", "age", "floor", "wing", "clinical_severity", "ct_value"]]

## 題目 9 解答

In [ ]:
import numpy as np

rng = np.random.default_rng(202)

n = 500
report_dates = pd.date_range("2026-03-01", periods=21, freq="D")

vaccination_status = rng.choice(
    ["未接種", "部分接種", "完整接種"], size=n, p=[0.30, 0.25, 0.45]
)
age = rng.integers(1, 90, size=n)
sex = rng.choice(["M", "F"], size=n)
report_date = report_dates[rng.integers(0, len(report_dates), size=n)]
test_result = rng.choice(["positive", "negative"], size=n, p=[0.55, 0.45])

# 依接種狀態設定臨床嚴重度分布（未接種者重症比例較高）
sev_categories = ["none", "mild", "moderate", "severe"]
sev_probs = {
    "未接種": [0.35, 0.35, 0.20, 0.10],
    "部分接種": [0.55, 0.30, 0.11, 0.04],
    "完整接種": [0.70, 0.25, 0.04, 0.01],
}
severity = np.array(
    [rng.choice(sev_categories, p=sev_probs[v]) for v in vaccination_status]
)

hospitalized = np.isin(severity, ["moderate", "severe"]).astype(int)
extra_hosp = (severity == "mild") & (rng.random(n) < 0.05)
hospitalized = np.where(extra_hosp, 1, hospitalized)

covid_df = pd.DataFrame({
    "report_id": [f"C{i:04d}" for i in range(1, n + 1)],
    "report_date": report_date,
    "age": age,
    "sex": sex,
    "vaccination_status": vaccination_status,
    "test_result": test_result,
    "severity": severity,
    "hospitalized": hospitalized,
})

# 1. 篩選陽性病例
covid_cases = covid_df.query("test_result == 'positive'").copy()
print(f"陽性病例數：{len(covid_cases)}")

# 2-3. 各接種狀態病例數與住院數，計算住院比例
vacc_stats = (
    covid_cases.groupby("vaccination_status")
    .agg(n_cases=("report_id", "count"), n_hosp=("hospitalized", "sum"))
    .reset_index()
)
vacc_stats["hospitalization_rate_pct"] = (
    vacc_stats["n_hosp"] / vacc_stats["n_cases"] * 100
).round(1)
vacc_stats = vacc_stats.sort_values("hospitalization_rate_pct", ascending=False)
print("\n=== 各接種狀態病例數與住院比例 ===")
print(vacc_stats)

# 4. 臨床嚴重度分布
print("\n=== 臨床嚴重度次數分布 ===")
print(covid_cases["severity"].value_counts())
print("\n=== 臨床嚴重度百分比分布 ===")
print(covid_cases["severity"].value_counts(normalize=True).mul(100).round(1))

# 5. 流行曲線
daily_counts = covid_cases.groupby("report_date").size().rename("cases")
full_range = pd.date_range(daily_counts.index.min(), daily_counts.index.max())
daily_counts = daily_counts.reindex(full_range, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(daily_counts.index, daily_counts.values, width=0.8, color="#D97757")
ax.set_title("社區篩檢站 COVID-19 每日新增陽性病例")
ax.set_xlabel("通報日期")
ax.set_ylabel("新增陽性病例數")
fig.autofmt_xdate(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 6. 解讀
top_group = vacc_stats.iloc[0]
full_vacc = vacc_stats.loc[
    vacc_stats["vaccination_status"] == "完整接種", "hospitalization_rate_pct"
].iloc[0]
print(
    f"\n解讀：{top_group['vaccination_status']}者的住院比例最高"
    f"（{top_group['hospitalization_rate_pct']}%），完整接種者住院比例僅 {full_vacc}%，"
    "顯示疫苗接種與降低重症、住院風險有關，支持持續推動疫苗接種與追加劑政策。"
)

## 題目 10 解答

In [ ]:
import numpy as np

rng = np.random.default_rng(818)

region_population = {
    "高雄市三民區": 34000,
    "高雄市苓雅區": 28000,
    "台南市北區": 21000,
    "台南市安南區": 19000,
    "屏東市": 15000,
}
regions = list(region_population.keys())

n_cases = 360
# 各區病例負擔權重（三民、安南積水容器較多，風險偏高，權重不與人口成比例）
region_weights = [0.34, 0.10, 0.14, 0.32, 0.10]

case_region = rng.choice(regions, size=n_cases, p=region_weights)
onset_date = pd.Timestamp("2026-06-01") + pd.to_timedelta(
    rng.integers(0, 56, size=n_cases), unit="D"
)  # 8 週監測期
age = rng.integers(1, 95, size=n_cases)
sex = rng.choice(["M", "F"], size=n_cases)

dengue_df = pd.DataFrame({
    "case_id": [f"D{i:04d}" for i in range(1, n_cases + 1)],
    "region": case_region,
    "symptom_onset_date": onset_date,
    "age": age,
    "sex": sex,
})

# 1. 各區病例數與平均年齡
region_stats = (
    dengue_df.groupby("region")
    .agg(n_cases=("case_id", "size"), mean_age=("age", "mean"))
    .reset_index()
)
region_stats["mean_age"] = region_stats["mean_age"].round(1)

# 2. 併入人口數，計算每十萬人發生率
pop_df = pd.DataFrame({
    "region": list(region_population.keys()),
    "population": list(region_population.values()),
})
region_stats = pd.merge(region_stats, pop_df, on="region", how="left")
region_stats["incidence_per_100k"] = (
    region_stats["n_cases"] / region_stats["population"] * 100_000
).round(1)
region_stats = region_stats.sort_values("incidence_per_100k", ascending=False)

# 3. epi_week 衍生欄位
dengue_df["epi_week"] = dengue_df["symptom_onset_date"].dt.isocalendar().week

# 4. 各區每週病例數 pivot table
weekly = (
    dengue_df.groupby(["region", "epi_week"])
    .size()
    .rename("n_cases")
    .reset_index()
)
weekly_pivot = weekly.pivot_table(
    index="region", columns="epi_week", values="n_cases", fill_value=0
)
print("=== 各區每週病例數 pivot table ===")
print(weekly_pivot)

# 各區病例數最多的那一週
peak_week = weekly.loc[weekly.groupby("region")["n_cases"].idxmax()][
    ["region", "epi_week"]
].rename(columns={"epi_week": "peak_week"})
region_stats = pd.merge(region_stats, peak_week, on="region", how="left")

print("\n=== 各區病例數、人口、每十萬人發生率與病例高峰週 ===")
print(region_stats)

# 5. 排序長條圖並標註高峰週
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=region_stats, x="region", y="incidence_per_100k",
    hue="region", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("各區登革熱每十萬人發生率（8 週監測期）")
ax.set_xlabel("行政區")
ax.set_ylabel("發生率（每十萬人）")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

for i, row in enumerate(region_stats.itertuples()):
    ax.text(
        i, row.incidence_per_100k + 5,
        f"{row.incidence_per_100k}\n(高峰第{row.peak_week}週)",
        ha="center", fontsize=8,
    )

plt.tight_layout()
plt.show()

# 6. 解讀
top_region = region_stats.iloc[0]
most_cases_region = region_stats.sort_values("n_cases", ascending=False).iloc[0]
print(
    f"\n解讀：{top_region['region']}每十萬人發生率最高（{top_region['incidence_per_100k']}）。"
)
if top_region["region"] == most_cases_region["region"]:
    print(
        f"與病例數最多的區（{most_cases_region['region']}）一致，"
        "代表該區病例負擔與人口規模皆偏高。"
    )
else:
    print(
        f"但病例數最多的是{most_cases_region['region']}（{most_cases_region['n_cases']} 例）。"
        f"{top_region['region']}雖然病例數不是最多，但因人口基數較小，實際發生率反而更高，"
        "顯示只看病例數容易忽略人口分母的影響，這也是為什麼流行病學監測要用發生率而非原始病例數比較地區風險。"
    )